In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score

# === Paths ===
ROOT_DIR = r"C:/Users/sanja/Desktop/EEG_Project/EEG/eeg/derivatives"
IMG_DIR = os.path.join(ROOT_DIR, "Spectrograms")
LABEL_CSV = os.path.join(ROOT_DIR, "labels.csv")

# === Hyperparameters ===
IMG_SIZE = (150, 150)
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", DEVICE)

# === Dataset Class ===
class EEGSpectrogramDataset(Dataset):
    def __init__(self, img_dir, labels_df, transform=None):
        self.img_dir = img_dir
        self.labels_df = labels_df
        self.transform = transform
        self.samples = []
        label_dict = dict(zip(labels_df["subject"], labels_df["label"]))

        for subj in os.listdir(img_dir):
            subj_path = os.path.join(img_dir, subj)
            if not os.path.isdir(subj_path):
                continue
            label = label_dict.get(subj)
            if label is None:
                continue
            label_num = 1 if label.lower() == "ad" else 0
            for img_file in os.listdir(subj_path):
                if img_file.endswith(".png"):
                    self.samples.append((os.path.join(subj_path, img_file), label_num, subj))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, subj = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32), subj

# === CNN Model (Paper-based) ===
class PaperCNN(nn.Module):
    def __init__(self):
        super(PaperCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Dropout(0.25),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE[0] // 8) * (IMG_SIZE[1] // 8), 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# === Training & Evaluation ===
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    for X, y, _ in dataloader:
        X, y = X.to(DEVICE), y.unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader):
    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for X, y, _ in dataloader:
            X, y = X.to(DEVICE), y.unsqueeze(1).to(DEVICE)
            preds = model(X)
            preds = (preds > 0.5).float()
            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(y.cpu().numpy())
    return accuracy_score(labels_list, preds_list)

# === Dataset Load ===
labels_df = pd.read_csv(LABEL_CSV)
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
])

dataset = EEGSpectrogramDataset(IMG_DIR, labels_df, transform=transform)
groups = np.array([subj for _, _, subj in dataset.samples])
unique_subjects = np.unique(groups)

# === LOSO Cross-Validation ===
accuracies = []

print("🧠 Starting LOSO Cross-Validation...\n")
for subj_test in tqdm(unique_subjects):
    test_idx = np.where(groups == subj_test)[0]
    train_idx = np.where(groups != subj_test)[0]

    train_subset = torch.utils.data.Subset(dataset, train_idx)
    test_subset = torch.utils.data.Subset(dataset, test_idx)

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)

    model = PaperCNN().to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, criterion, optimizer)

    acc = evaluate(model, test_loader)
    accuracies.append(acc)
    print(f"Subject {subj_test}: Accuracy = {acc:.3f}")

print(f"\n✅ Final LOSO Accuracy: {np.mean(accuracies):.3f} ± {np.std(accuracies):.3f}")


✅ Using device: cpu
🧠 Starting LOSO Cross-Validation...



  0%|          | 0/65 [00:00<?, ?it/s]

Subject sub-001: Accuracy = 1.000
Subject sub-002: Accuracy = 0.654
Subject sub-003: Accuracy = 0.000
Subject sub-004: Accuracy = 1.000
Subject sub-005: Accuracy = 0.769
Subject sub-006: Accuracy = 0.000
Subject sub-007: Accuracy = 1.000
Subject sub-008: Accuracy = 0.577
Subject sub-009: Accuracy = 1.000
Subject sub-010: Accuracy = 0.233
Subject sub-011: Accuracy = 0.320
Subject sub-012: Accuracy = 0.345
Subject sub-013: Accuracy = 0.929
Subject sub-014: Accuracy = 0.935
Subject sub-015: Accuracy = 0.867
Subject sub-016: Accuracy = 0.094
Subject sub-017: Accuracy = 0.964
Subject sub-018: Accuracy = 1.000
Subject sub-019: Accuracy = 0.267
Subject sub-020: Accuracy = 1.000
Subject sub-021: Accuracy = 0.900
Subject sub-022: Accuracy = 0.185
Subject sub-023: Accuracy = 0.464
Subject sub-024: Accuracy = 1.000
Subject sub-025: Accuracy = 0.000
Subject sub-026: Accuracy = 1.000
Subject sub-027: Accuracy = 0.963
Subject sub-028: Accuracy = 1.000
Subject sub-029: Accuracy = 1.000
Subject sub-03